# 🏗️ Notebook 03: 特征工程 — 让模型"看懂"电力数据

## Feature Engineering: Teaching ML Models to Understand Power Data

## 学习目标 (Learning Objectives)
1. 理解特征工程在机器学习管道中的角色
2. 掌握时间序列的常见特征类型
3. 理解"渐进式"特征工程的理念
4. 学会用 sin/cos 循环编码处理周期性特征
5. 理解 15 分钟粒度下 lag 特征的含义

## 什么是特征工程？(What is Feature Engineering?)

**特征工程 = 将领域知识编码为数字，让模型能看到规律。**

一个例子:
> 你告诉 XGBoost 原始数据是 `2024-08-15 14:30 | 52000 MW`。
> XGBoost 不理解下午 2:30、不理解 8 月、不理解周六。
> 特征工程转换后: `hour=14, day_of_week=5, month=8, is_weekend=1`。
> 现在 XGBoost 可以学到: 夏季周末下午的负荷低于工作日。

**原始数据 vs 特征数据 (15min 粒度):**
```
timestamp           load_mw   →   hour  month  dow  weekend  lag_24h  lag_168h
2024-08-15 14:15    51800    →   14    8      3    0        51600    51400
2024-08-15 14:30    51900    →   14    8      3    0        51700    51500
2024-08-15 14:45    52000    →   14    8      3    0        51800    51600
```
注意: `lag_24h` 在 15 分钟粒度下是 96 步前 (不是 24 步)，`lag_168h` 是 672 步前。

## 渐进式特征工程 (Progressive Feature Engineering)

我们采用三层金字塔式设计:

```
        ┌─────┐
        │Tier3│ 高级: rolling 统计, sin/cos 循环编码
        ├─────┤
        │Tier2│ 中级: 节假日, 周滞后 (lag_168h)
        ├─────┤
        │Tier1│ 核心: 小时/星期/月份/周末/日滞后 (lag_24h)
        └─────┘
```

这个设计的精髓:
1. Tier 1 跑通 XGBoost → 建立 baseline (5 个特征)
2. Tier 2 看到效果提升 → 理解增加特征 = 增加信息
3. Tier 3 尝试优化 → 理解特征工程的边际收益

In [ ]:
from ellectric.pipeline.data_loader import create_loader
from ellectric.pipeline.cleaner import clean_data
from ellectric.pipeline.features import FeatureEngineer
from ellectric.config import TimeConfig
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

# 验证 15min 配置
print(f"TimeConfig: {TimeConfig.points_per_day} 点/天, {TimeConfig.points_per_week} 点/周, {TimeConfig.freq}")

In [ ]:
# ── 加载并清洗数据 ──────────────────────────────
loader = create_loader('shandong')
df = loader.load_data('2024-06-01', '2024-09-30')
df = clean_data(df)

print(f"基础数据: {len(df)} 行 × {len(df.columns)} 列")
print(f"核心列: timestamp, load_mw, rt_price, wind_actual_mw, solar_actual_mw, ...")
print(f"时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
print(f"粒度: 15 分钟 (96 点/天)")
print(f"\n前 5 行:")
df[['timestamp', 'load_mw', 'rt_price', 'is_holiday', 'is_weekend']].head()

---
## Tier 1: 核心特征 (Core Features)

5 个特征: `hour`, `day_of_week`, `month`, `is_weekend`, `lag_24h`

### 每个特征的物理含义

| 特征 | 含义 | 为什么有效 |
|------|------|-----------|
| `hour` | 一天中的时刻 (0-23) | 捕获日内日周期——凌晨低谷、早晚高峰 |
| `day_of_week` | 星期几 (0=周一) | 工作日 vs 周末负荷模式不同 |
| `month` | 月份 (1-12) | 夏季制冷 vs 冬季取暖的季节性差异 |
| `is_weekend` | 是否周末 (0/1) | 工业负荷周末明显下降 |
| `lag_24h` | 24 小时前负荷 | 昨天同时刻是今天的最好估计 (持续法) |

### 15 分钟粒度下的关键区别

`lag_24h` = `df['load_mw'].shift(96)` — 后退 **96 个数据点**才是 24 小时前！

TimConfig 已配置为 `points_per_day=96`，所以 FeatureEngineer 自动使用正确的偏移量。

In [ ]:
# ── 添加 Tier 1 特征 ────────────────────────────
engineer = FeatureEngineer()
df_t1 = engineer.add_tier1_features(df)
t1_cols = engineer.get_feature_columns('tier1')

print(f"Tier 1 特征列: {t1_cols}")
print(f"特征数: {len(t1_cols)} 个")
print(f"\n带特征的 DataFrame 形状: {df_t1.shape}")
print(f"\n前 10 行 (每行 = 15 分钟):")
df_t1[t1_cols + ['load_mw']].head(10)

In [ ]:
# ── 特征可视化: is_weekend 的负荷差异 ───────────
weekday_mean = df_t1.groupby('hour')['load_mw'].mean()
weekend_mask = df_t1['is_weekend'] == 1
weekend_mean = df_t1[weekend_mask].groupby('hour')['load_mw'].mean()
weekday_only = df_t1[~weekend_mask].groupby('hour')['load_mw'].mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=weekday_only.index, y=weekday_only.values,
                         mode='lines+markers', name='工作日',
                         line=dict(color='#1f77b4', width=2)))
fig.add_trace(go.Scatter(x=weekend_mean.index, y=weekend_mean.values,
                         mode='lines+markers', name='周末',
                         line=dict(color='#ff7f0e', width=2, dash='dash')))

fig.update_layout(
    title='工作日 vs 周末 — 平均小时负荷 (2024 夏季)',
    xaxis=dict(title='小时', tickmode='linear', dtick=2),
    yaxis_title='平均负荷 (MW)',
    hovermode='x unified',
    height=400
)
fig.show()

# ── lag_24h 相关性 ────────────────────────────────
print(f"\n═══ lag_24h vs load_mw 相关性 ═══")
corr = df_t1['lag_24h'].corr(df_t1['load_mw'])
print(f"皮尔逊相关系数: {corr:.4f}")
print(f"\n📖 解读: r={corr:.3f} 说明 24 小时前的负荷与当前负荷高度相关。")
print("这是'持续法'有效的数学依据——电力负荷的日自相关性很强。")

---
## Tier 2: 中级特征 (Intermediate Features)

新增: `is_holiday`, `lag_168h`

- **is_holiday**: 节假日工业负荷断崖式下跌。注意: 山东数据已包含此列（从 CSV 直接读取），
  所以 FeatureEngineer 会优先使用已有列，不会覆盖。
- **lag_168h**: 168 小时前 (= 7 天 = 672 个 15min 点) 的负荷。
  如果今天是周三下午 2:15，上周三下午 2:15 的负荷很有参考价值。

In [ ]:
# ── 添加 Tier 2 特征 ────────────────────────────
df_t2 = engineer.add_tier2_features(df_t1)
t2_cols = engineer.get_feature_columns('tier2')

print(f"Tier 2 特征列 ({len(t2_cols)} 个): {t2_cols}")
print(f"新增: is_holiday, lag_168h")

# 检查节假日分布
holiday_count = df_t2['is_holiday'].sum()
print(f"\n节假日数据点: {holiday_count} / {len(df_t2)} ({holiday_count/len(df_t2)*100:.1f}%)")

# 节假日 vs 非节假日 负荷对比
holiday_mean = df_t2[df_t2['is_holiday'] == 1]['load_mw'].mean()
non_holiday_mean = df_t2[df_t2['is_holiday'] == 0]['load_mw'].mean()
print(f"节假日平均负荷: {holiday_mean:,.0f} MW")
print(f"非节假日平均负荷: {non_holiday_mean:,.0f} MW")
print(f"节假日负荷下降: {(1 - holiday_mean/non_holiday_mean)*100:.1f}%")

df_t2[t2_cols].head()

---
## Tier 3: 高级特征 — 循环编码 (Cyclic Encoding)

### 为什么用 sin/cos 编码小时？

如果直接把 `hour` 作为数值特征:
```
hour=0  (午夜)     和  hour=23 (晚上11点)  的距离是 |23-0| = 23
hour=23 (晚上11点) 和  hour=0  (午夜)      的距离是 |0-23| = 23
```
但午夜和晚上 11 点实际上只相隔 **1 小时**！

用 sin/cos 编码后:
```
hour_sin = sin(2π × hour / 24)
hour_cos = cos(2π × hour / 24)
```
这样 hour=23 和 hour=0 在 (sin, cos) 空间中**相邻**！

两个函数同时使用才能唯一确定每个小时在单位圆上的位置:
- 单独的 sin 不能区分 hour=6 (sin=1) 和 hour=18 (sin=-1)
- cos(6)=0, cos(18)=0 → sin 加了 cos 就能唯一确定

这称为**循环特征编码 (Cyclic Feature Encoding)**，
是处理周期性变量（时间、角度、风向、季节）的标准方法。

In [ ]:
# ── 添加 Tier 3 特征 ────────────────────────────
df_t3 = engineer.add_tier3_features(df_t2)
t3_cols = engineer.get_feature_columns('tier3')

print(f"Tier 3 特征列 ({len(t3_cols)} 个): {t3_cols}")
print(f"\n新增 4 个特征:")
print(f"  rolling_mean_24h — 过去 24h (96 点) 滚动均值")
print(f"  rolling_std_24h  — 过去 24h 滚动标准差 (波动性)")
print(f"  hour_sin         — 小时的 sin 编码")
print(f"  hour_cos         — 小时的 cos 编码")
df_t3[['hour', 'hour_sin', 'hour_cos', 'rolling_mean_24h', 'rolling_std_24h', 'load_mw']].head(10)

In [ ]:
# ── 可视化 sin/cos 循环编码 ────────────────────
import numpy as np

hours = np.arange(24)
hour_sin = np.sin(2 * np.pi * hours / 24)
hour_cos = np.cos(2 * np.pi * hours / 24)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('sin/cos vs 小时', '单位圆上的 24 小时'),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}]]
)

# 子图1: sin/cos 曲线
fig.add_trace(go.Scatter(x=hours, y=hour_sin, mode='lines+markers',
                         name='hour_sin', line=dict(color='#1f77b4', width=2)),
              row=1, col=1)
fig.add_trace(go.Scatter(x=hours, y=hour_cos, mode='lines+markers',
                         name='hour_cos', line=dict(color='#ff7f0e', width=2)),
              row=1, col=1)

# 子图2: 单位圆
fig.add_trace(go.Scatter(x=hour_sin, y=hour_cos, mode='markers+text',
                         text=[str(h) for h in hours],
                         textposition='top center',
                         marker=dict(size=10, color=hours, colorscale='Viridis'),
                         name='小时', showlegend=False),
              row=1, col=2)

fig.update_layout(
    title='sin/cos 循环编码 — 让 23 和 0 在数学上相邻',
    height=450,
    showlegend=True
)
fig.update_xaxes(title='小时', tickmode='linear', dtick=2, row=1, col=1)
fig.update_yaxes(title='值', row=1, col=1)
fig.update_xaxes(title='hour_sin', range=[-1.3, 1.3], row=1, col=2)
fig.update_yaxes(title='hour_cos', range=[-1.3, 1.3], row=1, col=2)
fig.show()

print("\n📖 解读: 注意看单位圆——hour=23 和 hour=0 在圆上紧挨着！")
print("这解决了'23 和 0 相距 23 个数值单位但在时间上只差 1 小时'的矛盾。")

---
## 特征汇总 (Feature Summary)

| Tier | 特征 | 15min 粒度含义 | 类型 |
|------|------|---------------|------|
| T1 | `hour` | 小时 (0-23) | 时间 |
| T1 | `day_of_week` | 星期几 (0=周一) | 时间 |
| T1 | `month` | 月份 (1-12) | 时间 |
| T1 | `is_weekend` | 是否周末 | 布尔 |
| T1 | `lag_24h` | 96 步前 (24h) 负荷 | 滞后 |
| T2 | `is_holiday` | 是否节假日 | 布尔 |
| T2 | `lag_168h` | 672 步前 (168h) 负荷 | 滞后 |
| T3 | `rolling_mean_24h` | 过去 96 点均值 | 滚动 |
| T3 | `rolling_std_24h` | 过去 96 点标准差 | 滚动 |
| T3 | `hour_sin` | sin(2π×hour/24) | 循环 |
| T3 | `hour_cos` | cos(2π×hour/24) | 循环 |

**关键**: 所有 scaler 在 forecaster 内部按 fold 独立 fit——不在全量数据上调 `fit_transform()`！

**下一步: Notebook 04 → XGBoost 训练和评估**

### 思考题 (Reflection Questions)

1. 在 15 分钟粒度下，`lag_96` (= 24h 前) 和 `lag_672` (= 7 天前) 哪个对预测更有用？为什么？
2. `rolling_std_24h` (过去 24h 波动性) 告诉模型什么信息？预测负荷时波动性大意味着什么？
3. 如果加入 `month_sin` / `month_cos` (月份循环编码)，为什么 12 个月也需要循环编码？